# Cygnus: evidencia de forecasting comercial

Ejecutar desde el entorno del repositorio. La demo es sintética y no acredita precisión en Cygnus. Para datos reales ejecutar scripts/64_forecasting_medallio.bat y sustituir RUN_DIR por la carpeta resultante.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from replica_cygnus.commercial_forecasting.core import Config
from replica_cygnus.commercial_forecasting.service import execute, synthetic_panel
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'scripts/commercial_forecasting.py').exists())
RUN_DIR, manifest, forecasts, backtest = execute(synthetic_panel(), ROOT, ROOT/'artifacts/commercial_forecasting', Config(), synthetic=True)
print(manifest['evidence_level'], RUN_DIR)

## Qué se entrenó y con qué datos
Fingerprint, versión, cortes y configuración. Los IDs de estados son locales a esta ejecución; no corresponden automáticamente a nombres comerciales.

In [ ]:
manifest = json.loads((RUN_DIR/'manifest.json').read_text())
manifest['final_training']

## Prueba final y comparación
La selección usa trayectorias completas de validación y controles por proyecto. Comparar candidatos sobre los mismos casos; metrics.csv puede tener poblaciones distintas. El histórico ya inspeccionado sigue siendo diagnóstico de desarrollo. Error en unidades, no una tasa de acierto.

In [ ]:
paired = pd.read_csv(RUN_DIR/'paired_comparisons.csv')
paired[paired.partition.eq('test') & paired.horizon.isin([0,1,3,6])]


## Política ejecutable, cobertura e integridad
La referencia se mantiene donde no hay evidencia suficiente. La cobertura se mide también sobre el stock, no solo contando proyectos. Los cortes solapados no son ensayos independientes.

In [ ]:
from replica_cygnus.commercial_forecasting.robustness import verify_integrity
robustness = json.loads((RUN_DIR/'robustness.json').read_text())
print('Integridad:', verify_integrity(RUN_DIR)['status'])
print('Soporte temporal:', robustness['test_temporal_uncertainty'])
print('Política evaluada:', robustness['test_policy_score'])
pd.read_csv(RUN_DIR/'project_coverage.csv')


## Predicciones nuevas y stock
Horizontes acumulados: no sumar 1, 3 y 6. Filtrar modelo seleccionado antes de agregar.

In [ ]:
forecasts = pd.read_csv(RUN_DIR/'predictions.csv')
forecasts[forecasts.is_selected & forecasts.horizon.isin([1,3,6])]

## Explicación del GMM y del bosque
Probabilidades de pertenencia no equivalen a probabilidades de venta. Relevancias del bosque describen uso de variables, no causalidad.

In [ ]:
profiles = json.loads((RUN_DIR/'model_profiles.json').read_text())
print('Estados GMM:', profiles['clusters'])
print('Relevancias:', profiles['feature_importance'])

## Hacerlo útil hoy
Registrar metas y acciones con commercial_forecasting.py. Consumir vistas Power BI descritas en docs/COMMERCIAL_FORECASTING.md. Reportar evidencia prospectiva únicamente cuando los resultados maduren.